# Setup

## Imports

In [44]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os
from typing import Literal

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point, box
from shapely.ops import unary_union

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer
from src.utils.vis.map_special import (
    build_map_building_basic,
    build_map_point_floorspace_demand_type,
    build_map_point_numeric_and_category
)

# configs
sns.set_theme()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Overview

Only coordinates and number of units will be used

the data includes `osm_id` referring to the landuse area that corresponds to the base, 
but I can't be arsed to make use of that

active + 1/6 reserve???

Final produced GDF:
* `name`
* `workers`
* `military_type` - branch
* `geometry`

# Load Military Data and Clean

In [45]:
# (active, reserve)
BRANCHES = {
    "army": (40000, 240000),
    "navy": (4000, 5000),
    "airforce": (6000, 7500),
}

In [46]:
fpath = os.path.join(data_filepaths.path_sd_military, "bases.json")

bases_df = pd.read_json(fpath)

In [47]:
bases_df.head()

,name,branch,unit_count,units,osm_id,lat,lon
0,Amoy Quee Camp,army,2,"[HQ National Cadet Corps (HQ NCC), HQ 3rd Divi...",469963896,1.386331,103.853277
1,Bedok Camp,army,3,"[HQ 7th Singapore Infantry Brigade (HQ 7 SIB),...",312723415,1.317111,103.954219
2,Clementi Camp,army,4,"[HQ 2nd People's Defence Force (HQ 2 PDF), Isl...",75049904,1.330969,103.762377
3,Depot Road Camp,army,1,[Defence Science and Technology Agency (DSTA)],1136399093,1.280104,103.815463
4,Dieppe Barracks,army,2,"[HQ Guards (HQ GDS), 1st Battalion, Singapore ...",390777714,1.421859,103.823150


In [48]:
def process_loaded_bases_df_to_gdf(bases_df: pd.DataFrame, crs=3414):
    bases_df = bases_df.copy()
    geometry = bases_df[["lon", "lat"]].apply(
        lambda r: Point(r["lon"], r["lat"]), axis=1
    )

    bases_gdf = (
        gpd.GeoDataFrame(bases_df, geometry=geometry, crs=4326)
        .drop(columns=["lat", "lon"])
        .to_crs(crs)
    )

    return bases_gdf


bases_gdf = process_loaded_bases_df_to_gdf(bases_df)

In [49]:
for branch, personnel in BRANCHES.items():
    branch_bases = bases_gdf[bases_gdf["branch"] == branch]

    # active + 1/5 reserve
    total_personnel = personnel[0] + 1 / 6 * personnel[1]

    unit_counts = branch_bases["unit_count"]

    if branch == "army":
        # custom unit count multipliers with "HQ" deemphasized
        hq_count = branch_bases["units"].map(lambda l: len([x for x in l if "HQ" in x]))
        unit_counts -= 1 / 2 * hq_count # HQ treated as half-weight

    worker_count = unit_counts / unit_counts.sum() * total_personnel

    bases_gdf.loc[bases_gdf["branch"] == branch, "workers"] = worker_count

In [50]:
bases_gdf.head()

,name,branch,unit_count,units,osm_id,geometry,workers
0,Amoy Quee Camp,army,2,"[HQ National Cadet Corps (HQ NCC), HQ 3rd Divi...",469963896,POINT (30221.115 40918.963),800.0
1,Bedok Camp,army,3,"[HQ 7th Singapore Infantry Brigade (HQ 7 SIB),...",312723415,POINT (41455.051 33265.286),2000.0
2,Clementi Camp,army,4,"[HQ 2nd People's Defence Force (HQ 2 PDF), Isl...",75049904,POINT (20104.934 34797.42),2800.0
3,Depot Road Camp,army,1,[Defence Science and Technology Agency (DSTA)],1136399093,POINT (26012.819 29172.923),800.0
4,Dieppe Barracks,army,2,"[HQ Guards (HQ GDS), 1st Battalion, Singapore ...",390777714,POINT (26868.385 44847.463),1200.0


## Vis

In [51]:
build_map_point_numeric_and_category(
    bases_gdf,
    numeric_col="workers",
)

# Export

In [ ]:
def export(bases_gdf, fpath: str = data_filepaths.MILITARY_WORKFORCE):
    os.makedirs(os.path.dirname(fpath) or ".", exist_ok=True)

    bases_gdf = bases_gdf.copy()
    bases_gdf = bases_gdf.rename(columns={"branch": "military_type"})

    bases_gdf = bases_gdf[["name", "workers", "military_type", "geometry"]]

    bases_gdf.to_parquet(fpath)
    return bases_gdf


export(bases_gdf).head()